In [163]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [164]:
class BatsmanState(TypedDict):
    runs:int
    balls:int
    fours: int
    sixes: int
    sr:float
    balls_per_boundary:float
    boundary_percent:float
    summary:str

In [165]:
def cal_sr(state:BatsmanState):
    sr=(state['runs']/state['balls'])*100
    return {'sr':sr}  

def cal_bpb(state:BatsmanState):
    bpb=(state['balls']/(state['fours']+state['sixes']))
    return {'balls_per_boundary':bpb}

def boundary_percent(state:BatsmanState):
    bp=(((state['sixes']*6)+(state['fours'])*4)/state['balls'])*100
    return {'boundary_percent':bp}

def summary(state:BatsmanState):
    summary= f"""
Strike Rate = {state['sr']} \n
Balls per boundary = {state['balls_per_boundary']} \n
Boundary percent - {state['boundary_percent']} \n
"""
    return {'summary':summary}

In [166]:
graph=StateGraph(BatsmanState)

graph.add_node('sr',cal_sr)
graph.add_node('balls_per_boundary',cal_bpb)
graph.add_node('boundary_percent',boundary_percent)
graph.add_node('summary',summary)
graph.add_edge(START,'sr')
graph.add_edge(START,'balls_per_boundary')
graph.add_edge(START,'boundary_percent')
graph.add_edge('sr','summary')
graph.add_edge('balls_per_boundary','summary')
graph.add_edge('boundary_percent','summary')
graph.add_edge('summary',END)

workflow=graph.compile()
 

In [ ]:
initial_state={
     "runs":100,
    "balls":50,
    "fours":6,
    "sixes":4,
}
workflow.invoke(initial_state)


In [168]:
#### ESSAY model
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

class Essay(TypedDict):
    input:str
    
    clarity_of_thought:int
    depth_of_analysis:int
    grammar:int
    final_score:int
    # remarks:str

In [169]:
def cot(state:Essay):
    input=state['input']
    prompt=f"rate the {input} from 1 to 10 on the basis of clarity of thought. ONLY return the number"
    output=llm.invoke(prompt).content
    return {'clarity_of_thought':int(output)}

def doa(state:Essay):
    input=state['input']
    prompt=f"rate the {input} from 1 to 10 on the basis of depth of analysis. ONLY return the number."
    output=llm.invoke(prompt).content
    return {'depth_of_analysis':int(output)}

def gr(state:Essay):
    input=state['input']
    prompt=f"rate the {input} from 1 to 10 on the basis of grammar. ONLY return the number."
    output=llm.invoke(prompt).content
    return {'grammar':int(output)}

def final(state:Essay):
    input=state['input']
    output=(state['clarity_of_thought']+state['depth_of_analysis']+state['grammar'])
    return {'final_score':int(output)}

In [170]:
graph=StateGraph(Essay)

graph.add_node('clarity_of_thought',cot)
graph.add_node('depth_of_analysis',doa)
graph.add_node('grammar',gr)
graph.add_node('final_score',final)
graph.add_edge(START,'clarity_of_thought')
graph.add_edge(START,'depth_of_analysis')
graph.add_edge(START,'grammar')
graph.add_edge('clarity_of_thought','final_score')
# graph.add_edge('clarity_of_thought','summary')
graph.add_edge('depth_of_analysis','final_score')
# graph.add_edge('depth_of_analysis','summary')
graph.add_edge('grammar','final_score')
# graph.add_edge('grammar','summary')
graph.add_edge('final_score',END)
workflow=graph.compile()

In [ ]:
initial_state={
    "input":"Goa is a land near sea. I love goa"
}
final_state=workflow.invoke(initial_state)
print(final_state)